# ⚠️ Notebook 4 — Conflict Detector
## What this notebook does
Compares obligations of the same type across all documents and finds contradictions.

**Simple explanation:**  
Imagine putting all the "uptime" clauses from every contract side by side.  
If one says 99.5% and another says 99.9% — that is a conflict.  
The AI reads them together and explains the contradiction.

**Technical explanation:**  
Groups obligations by type using SQL GROUP BY on ob_type.  
For each type with obligations from 2+ different documents,  
sends a cross-document comparison prompt to Groq.  
LLM performs Natural Language Inference (NLI) — specifically  
contradiction detection. Result stored in conflicts table with severity.


## Step 1 — Setup

In [ ]:
import os, json, re, time
from groq import Groq
from dotenv import load_dotenv
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from itertools import combinations
from datetime import datetime
import pandas as pd

load_dotenv()
client  = Groq(api_key=os.getenv("GROQ_API_KEY"))
engine  = create_engine("sqlite:///database/contractiq.db", echo=False)
Session = sessionmaker(bind=engine)
session = Session()

Base = declarative_base()

class Conflict(Base):
    __tablename__  = "conflicts"
    __table_args__ = {'extend_existing': True}
    id            = Column(Integer, primary_key=True)
    ob1_id        = Column(Integer)
    ob2_id        = Column(Integer)
    doc1_name     = Column(String(255))
    doc2_name     = Column(String(255))
    conflict_type = Column(String(100))
    ob1_text      = Column(Text)
    ob2_text      = Column(Text)
    explanation   = Column(Text)
    severity      = Column(String(20))
    created_at    = Column(DateTime, default=datetime.now)

Base.metadata.create_all(engine)
print("✅ Setup complete")

## Step 2 — Load and Group Obligations

**Simple:** Gather all obligations of the same type together.

**Technical:** SQL query with WHERE filter on ob_type. Groups returned as Python dicts keyed by ob_type. Only types with obligations from 2+ documents are candidates for conflict.

In [ ]:
from sqlalchemy import text as sql_text

# Load all obligations
rows = session.execute(sql_text(
    "SELECT id, doc_name, ob_type, text, clause_ref, risk_level FROM obligations ORDER BY ob_type"
)).fetchall()

obligations = [
    {"id": r[0], "doc_name": r[1], "ob_type": r[2],
     "text": r[3], "clause_ref": r[4], "risk_level": r[5]}
    for r in rows
]

# Group by obligation type
from collections import defaultdict
grouped = defaultdict(list)
for ob in obligations:
    grouped[ob["ob_type"]].append(ob)

print(f"📊 Total obligations loaded: {len(obligations)}")
print(f"\n📊 Grouped by type:")
for ob_type, items in sorted(grouped.items()):
    docs = set(item["doc_name"] for item in items)
    conflict_possible = "⚠️  CONFLICT POSSIBLE" if len(docs) > 1 else "✅ Single doc"
    print(f"   {ob_type:20s}: {len(items):3d} obligations | {len(docs)} docs | {conflict_possible}")

## Step 3 — The Conflict Detection Prompt

**Simple:** We ask the AI: 'Do these two clauses from different documents contradict each other?'

**Technical:** Textual entailment / contradiction detection. Prompt provides both clauses with document context. LLM performs cross-document NLI and returns structured verdict with explanation.

In [ ]:
CONFLICT_SYSTEM = """You are a senior legal expert specializing in contract conflict analysis.
You compare clauses from different documents and identify contradictions with precision.
Always respond with valid JSON only."""

def build_conflict_prompt(ob1, ob2):
    """
    Builds a conflict detection prompt for two obligations.
    
    Simple:    Shows the AI two clauses and asks if they disagree.
    Technical: Structured NLI prompt. Forces JSON verdict with
               explanation and severity. Guides the model to focus
               on semantic contradictions, not just word differences.
    """
    return f"""Compare these two clauses from different contract documents:

CLAUSE 1:
Document  : {ob1['doc_name']}
Clause Ref: {ob1['clause_ref']}
Text      : {ob1['text']}

CLAUSE 2:
Document  : {ob2['doc_name']}
Clause Ref: {ob2['clause_ref']}
Text      : {ob2['text']}

TASK: Determine if these clauses CONFLICT or CONTRADICT each other.
A conflict means: the same topic has DIFFERENT requirements/values in different documents,
which could cause legal ambiguity, financial exposure, or compliance risk.

Respond with this exact JSON:
{{
  "conflict_found": true or false,
  "conflict_type": "brief name of the conflict (e.g. Liability Cap, Uptime Target)",
  "explanation": "clear explanation of what is different and why it matters (2-3 sentences)",
  "severity": "HIGH", "MEDIUM", or "LOW",
  "recommendation": "which document should take precedence and why"
}}

JSON RESPONSE:"""

print("✅ Conflict detection prompt defined")

## Step 4 — Run Conflict Detection

This compares all pairs of same-type obligations from different documents.

⏳ May take 2-3 minutes — one Groq call per comparison pair.

In [ ]:
def detect_conflicts():
    total_conflicts = 0
    comparisons     = 0
    
    for ob_type, obs in grouped.items():
        # Get all unique documents for this type
        doc_obs = defaultdict(list)
        for ob in obs:
            doc_obs[ob["doc_name"]].append(ob)
        
        # Only check types that appear in multiple documents
        if len(doc_obs) < 2:
            continue
        
        print(f"\n🔍 Checking '{ob_type}' across {len(doc_obs)} documents...")
        
        # Get best representative obligation from each document
        doc_names   = list(doc_obs.keys())
        doc_best_ob = {}
        for doc, doc_ob_list in doc_obs.items():
            # Pick the longest/most detailed obligation
            doc_best_ob[doc] = max(doc_ob_list, key=lambda x: len(x["text"]))
        
        # Compare every pair of documents
        for doc1, doc2 in combinations(doc_names, 2):
            ob1 = doc_best_ob[doc1]
            ob2 = doc_best_ob[doc2]
            comparisons += 1
            
            print(f"   Comparing: {doc1[:30]} vs {doc2[:30]}...", end=" ")
            
            prompt = build_conflict_prompt(ob1, ob2)
            
            try:
                response = client.chat.completions.create(
                    model      = "llama3-70b-8192",
                    messages   = [
                        {"role": "system", "content": CONFLICT_SYSTEM},
                        {"role": "user",   "content": prompt}
                    ],
                    max_tokens  = 600,
                    temperature = 0.1,
                )
                
                raw = response.choices[0].message.content.strip()
                raw = re.sub(r'^```json\s*','',raw)
                raw = re.sub(r'^```\s*','',raw)
                raw = re.sub(r'\s*```$','',raw)
                
                result = json.loads(raw)
                
                if result.get("conflict_found"):
                    # Save conflict to DB
                    conflict = Conflict(
                        ob1_id        = ob1["id"],
                        ob2_id        = ob2["id"],
                        doc1_name     = doc1,
                        doc2_name     = doc2,
                        conflict_type = result.get("conflict_type", ob_type),
                        ob1_text      = ob1["text"],
                        ob2_text      = ob2["text"],
                        explanation   = result.get("explanation",""),
                        severity      = result.get("severity","MEDIUM"),
                    )
                    session.add(conflict)
                    session.commit()
                    total_conflicts += 1
                    print(f"⚠️  CONFLICT ({result.get('severity','?')})")
                else:
                    print("✅ No conflict")
                
                time.sleep(0.5)
                
            except json.JSONDecodeError:
                print("⚠️  Parse error")
            except Exception as e:
                if "rate_limit" in str(e).lower():
                    print("⏳ Rate limit — waiting 15s...")
                    time.sleep(15)
                else:
                    print(f"❌ {str(e)[:50]}")
    
    return total_conflicts, comparisons

total, comps = detect_conflicts()

print(f"\n{'='*50}")
print(f"✅ CONFLICT DETECTION COMPLETE")
print(f"   Comparisons made : {comps}")
print(f"   Conflicts found  : {total}")
print(f"{'='*50}")
print("\n▶ Run Notebook 5 next: RAG Engine")

## Step 5 — Review Detected Conflicts

In [ ]:
conflicts = session.execute(sql_text(
    "SELECT conflict_type, severity, doc1_name, doc2_name, substr(explanation,1,150) FROM conflicts ORDER BY severity"
)).fetchall()

if not conflicts:
    print("⚠️  No conflicts found yet. Make sure you processed multiple documents.")
else:
    print(f"⚠️  {len(conflicts)} CONFLICTS DETECTED:\n")
    for i, c in enumerate(conflicts, 1):
        print(f"  [{i}] {c[0]} | Severity: {c[1]}")
        print(f"       Doc 1: {c[2]}")
        print(f"       Doc 2: {c[3]}")
        print(f"       Note: {c[4]}...")
        print()